In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.image import resize
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
from tensorflow.keras.applications import VGG16

In [ ]:
# Install the gdown library to download the file from Google Drive
!pip install gdown

# The URL of your shared file
file_url = 'https://drive.google.com/file/d/1B4cfrzSkt7ZYAQ1PDl_TD1gthy82GHE7/view?usp=sharing'

# The local filename where the downloaded data will be saved
output_filename = 'hmnist_28_28_RGB.csv'

# Download the file using gdown
!gdown --id 1B4cfrzSkt7ZYAQ1PDl_TD1gthy82GHE7 -O "$output_filename"

# Now, you can initialize your DataFrame using the local file
df = pd.read_csv(output_filename)

# You can now proceed with the rest of your code
print(df.head())

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1B4cfrzSkt7ZYAQ1PDl_TD1gthy82GHE7
To: /content/hmnist_28_28_RGB.csv
100% 91.8M/91.8M [00:00<00:00, 287MB/s]
   pixel0000  pixel0001  pixel0002  pixel0003  pixel0004  pixel0005  \
0        192        153        193        195        155        192   
1         25         14         30         68         48         75   
2        192        138        153        200        145        163   
3         38         19         30         95         59         72   
4        158        113        139        194        144        174   

   pixel0006  pixel0007  pixel0008  pixel0009  ...  pixel2343  pixel2344  \
0        197        154        185        202  ...        173        124   
1        123         93        126      

In [ ]:
df.columns

Index(['pixel0000', 'pixel0001', 'pixel0002', 'pixel0003', 'pixel0004',
       'pixel0005', 'pixel0006', 'pixel0007', 'pixel0008', 'pixel0009',
       ...
       'pixel2343', 'pixel2344', 'pixel2345', 'pixel2346', 'pixel2347',
       'pixel2348', 'pixel2349', 'pixel2350', 'pixel2351', 'label'],
      dtype='object', length=2353)

In [ ]:
X = df.drop('label', axis=1).values
y = df['label']

In [ ]:
X = X.reshape(-1,28,28,3)

In [ ]:
X_resized = resize(X, size=(32, 32))

In [ ]:
X_resized_np = X_resized.numpy()

X_train, X_test, y_train, y_test = train_test_split(X_resized_np, y, test_size=0.2, random_state=42)

In [ ]:
TARGET_SIZE = 128
X_resized2 = resize(X, size=(TARGET_SIZE, TARGET_SIZE))

In [ ]:
X_resized_np2 = X_resized2.numpy()

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_resized_np2, y, test_size=0.2, random_state=42)

In [ ]:
csv_path = '/content/hmnist_28_28_RGB.csv'

df = pd.read_csv(csv_path)


X = df.drop('label', axis=1).values
y = df['label'].values
X = X.reshape(-1, 28, 28, 3)

TARGET_SIZE = 128
X_resized = tf.image.resize(X, [TARGET_SIZE, TARGET_SIZE]).numpy()


X_train, X_test, y_train, y_test = train_test_split(X_resized, y, test_size=0.2, random_state=42, stratify=y)


inputs = Input(shape=(TARGET_SIZE, TARGET_SIZE, 3))

vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(TARGET_SIZE, TARGET_SIZE, 3))
vgg_base.trainable = False # Freeze pre-trained layers


x = vgg_base(inputs, training=False)
x = Flatten()(x)
x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
outputs = Dense(7, activation='softmax')(x) # 7 output classes


model = Model(inputs, outputs)

model.summary()

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    fill_mode='nearest'
)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=25,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping]
)


--- Starting Model Training Process ---
✅ Data loaded successfully from /content/hmnist_28_28_RGB.csv
Resizing images from 28x28 to 128x128...
✅ Images resized.
Building model with Keras Functional API...
✅ Model built successfully.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg16 (Functional)              │ (None, 4, 4, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,814,919 (64.14 MB)

 Trainable params: 2,099,719 (8.01 MB)

 Non-trainable params: 14,715,200 (56.13 MB)


--- Starting Model Training ---


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 62s 210ms/step - accuracy: 0.5197 - loss: 1.8158 - val_accuracy: 0.7084 - val_loss: 0.8476
Epoch 2/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 40s 158ms/step - accuracy: 0.7074 - loss: 0.8836 - val_accuracy: 0.7289 - val_loss: 0.8198
Epoch 3/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 41s 163ms/step - accuracy: 0.7192 - loss: 0.7877 - val_accuracy: 0.7334 - val_loss: 0.7508
Epoch 4/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 40s 160ms/step - accuracy: 0.7341 - loss: 0.7334 - val_accuracy: 0.7404 - val_loss: 0.7052
Epoch 5/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 40s 158ms/step - accuracy: 0.7514 - loss: 0.6793 - val_accuracy: 0.7429 - val_loss: 0.6900
Epoch 6/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 40s 158ms/step - accuracy: 0.7518 - loss: 0.6657 - val_accuracy: 0.7449 - val_loss: 0.6815
Epoch 7/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 40s 160ms/step - accuracy: 0.7560 - loss: 0.6720 - val_accuracy: 0.7549 - val_loss: 0.6694
Epoch 8/25
251/251 ━━━━━━━━━━━━━━━━━━━━ 42s 164ms/step - accuracy: 0.7526 - loss: 0

✅ Model training complete.
✅ New model saved successfully to the Colab session as 'skin_cancer_detection_model_v2.h5'!
You can now download it from the 'Files' tab on the left.
